# Lab 04 — Bayesian Cosmology with a Real Pantheon+ Likelihood

**Format:** 4-hour hands-on practical  
**Status:** Formative + weekly checkpoint  
**Real data:** Pantheon+ / SH0ES  
**Covariance:** published statistical covariance

> ## Scientific question
> **How does a likelihood become a posterior, and how do nuisance calibration and prior choices propagate into cosmological uncertainty?**

We continue with the same real-data model:

\[
m_B^{\rm corr}
=
\mu(z;\Omega_m,H_0)+M_B+\epsilon,
\]

with \(H_0=70\) used only as a zero-point convention.

## 0. Five-line statement

**Estimand:**  
**Observed data:**  
**Generative assumptions:**  
**Uncertainty / selection:**  
**Validation / falsification test:**

In [ ]:
from pathlib import Path
import io, sys, time, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.linalg import cho_factor, cho_solve
from scipy.interpolate import RegularGridInterpolator
from scipy.special import logsumexp
from astropy.cosmology import FlatLambdaCDM

SEED = 2026
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
BASE = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/"
    "DataRelease/main/Pantheon%2B_Data/4_DISTANCES_AND_COVAR/"
)
URL_DATA = BASE + "Pantheon%2BSH0ES.dat"
URL_STAT = BASE + "Pantheon%2BSH0ES_STATONLY.cov"
URL_FULL = BASE + "Pantheon%2BSH0ES_STAT%2BSYS.cov"

CACHE = Path("data_cache/pantheon_plus")
CACHE.mkdir(parents=True, exist_ok=True)

def download_if_needed(url, path, timeout=180):
    path = Path(path)
    if not path.exists():
        print("Downloading", path.name, "...")
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        path.write_bytes(r.content)
    return path

def load_pantheon_table():
    p = download_if_needed(URL_DATA, CACHE/"Pantheon+SH0ES.dat")
    return pd.read_csv(p, sep=r"\s+")

def load_covariance(kind="STATONLY"):
    if kind.upper() == "STATONLY":
        url = URL_STAT
        path = CACHE/"Pantheon+SH0ES_STATONLY.cov"
    elif kind.upper() in {"FULL","STAT+SYS","STATSYS"}:
        url = URL_FULL
        path = CACHE/"Pantheon+SH0ES_STAT+SYS.cov"
    else:
        raise ValueError("kind must be STATONLY or FULL")

    p = download_if_needed(url, path)
    flat = np.loadtxt(p)
    n = int(flat[0])
    values = flat[1:]
    if values.size != n*n:
        raise ValueError(f"Expected {n*n} covariance entries, found {values.size}")
    return values.reshape(n, n)

sn = load_pantheon_table()
print("Pantheon+ rows:", len(sn))

In [ ]:
def teaching_mask(sn):
    """Real-data teaching sample: non-Cepheid calibrators with zHD > 0.01."""
    mask = (
        np.isfinite(sn["zHD"].to_numpy(float)) &
        np.isfinite(sn["m_b_corr"].to_numpy(float)) &
        (sn["zHD"].to_numpy(float) > 0.01) &
        (sn["IS_CALIBRATOR"].to_numpy(int) == 0)
    )
    return mask

def subset_data_cov(sn, cov, mask):
    idx = np.flatnonzero(mask)
    d = sn.iloc[idx].reset_index(drop=True).copy()
    c = cov[np.ix_(idx, idx)]
    return d, c, idx

def mu_flat_lcdm(z, omega_m, H0=70.0):
    cosmo = FlatLambdaCDM(H0=H0, Om0=omega_m, Tcmb0=2.725)
    return cosmo.distmod(np.asarray(z, float)).value

def coefficient_grid(data, cov, omega_grid, H0=70.0):
    """
    For model m_b_corr = mu(z; Omega_m,H0) + M_B,
    compute coefficients so that
    chi2(Omega_m, M_B) = a - 2 M_B b + M_B^2 d.
    """
    z = data["zHD"].to_numpy(float)
    m = data["m_b_corr"].to_numpy(float)
    one = np.ones(len(data))

    cf = cho_factor(cov, lower=True, check_finite=False)
    cinv_one = cho_solve(cf, one, check_finite=False)
    dcoef = one @ cinv_one

    a = np.empty(len(omega_grid))
    b = np.empty(len(omega_grid))

    for k, om in enumerate(omega_grid):
        y = m - mu_flat_lcdm(z, om, H0=H0)
        cinv_y = cho_solve(cf, y, check_finite=False)
        a[k] = y @ cinv_y
        b[k] = one @ cinv_y

    return a, b, float(dcoef)

def profile_mb(a, b, d):
    mb_hat = b/d
    chi2_prof = a - b*b/d
    return mb_hat, chi2_prof

def chi2_surface(a, b, d, mb_grid):
    mb = np.asarray(mb_grid, float)[None, :]
    return a[:,None] - 2*b[:,None]*mb + d*mb*mb

In [ ]:
cov_stat = load_covariance("STATONLY")
mask = teaching_mask(sn)
data, Cstat, original_indices = subset_data_cov(sn, cov_stat, mask)

OMEGA_GRID = np.linspace(0.10, 0.60, 101)
MB_GRID = np.linspace(-20.2, -18.2, 161)
H0_CONVENTION = 70.0

a, b, d = coefficient_grid(data, Cstat, OMEGA_GRID, H0=H0_CONVENTION)
CHI2 = chi2_surface(a, b, d, MB_GRID)

print("Posterior grid shape:", CHI2.shape)

# Part I — Bayes' theorem on a grid

## 1. Likelihood

Ignoring constants independent of \((\Omega_m,M_B)\),

\[
\ln\mathcal L
=
-\frac12\chi^2.
\]

## 2. Baseline priors

Use

\[
p(\Omega_m)=\mathrm{Uniform}(0.10,0.60),
\]

\[
p(M_B)=\mathrm{Uniform}(-20.2,-18.2).
\]

Within those bounds, the prior is constant; outside it is zero.

Therefore the posterior shape inside the grid is simply proportional to the likelihood.

In [ ]:
logL = -0.5 * CHI2

# Numerical normalization on the discrete grid.
logP = logL - logsumexp(logL)
P = np.exp(logP)
P /= P.sum()

print("Posterior grid normalization:", P.sum())

## 3. Plot the joint posterior

Use posterior contours that enclose approximately 68% and 95% of the discrete posterior probability.

A contour at a fixed density value is not automatically a 68% contour; the density threshold must be chosen so that the enclosed probability has the desired value.

In [ ]:
def density_thresholds(P, probs=(0.68,0.95)):
    flat = P.ravel()
    order = np.argsort(flat)[::-1]
    sorted_p = flat[order]
    csum = np.cumsum(sorted_p)
    thresholds = []
    for q in probs:
        j = np.searchsorted(csum, q)
        thresholds.append(sorted_p[j])
    return thresholds

thr68, thr95 = density_thresholds(P, (0.68,0.95))

fig, ax = plt.subplots(figsize=(7,5))
# contour levels must be increasing:
ax.contour(MB_GRID, OMEGA_GRID, P, levels=sorted([thr95,thr68]))
ax.set_xlabel(r"$M_B$")
ax.set_ylabel(r"$\Omega_m$")
ax.set_title("Pantheon+ posterior: statistical covariance")
plt.show()

### Exercise 1

Describe the orientation of the \((\Omega_m,M_B)\) posterior.

Is it strongly degenerate, weakly correlated, or nearly separable over this redshift range?

Do not infer this from a correlation coefficient alone—look at the geometry.

# Part II — Marginalization

## 4. Marginal posteriors

For the discrete grid,

\[
p(\Omega_m\mid D)
\propto
\sum_{M_B}
p(\Omega_m,M_B\mid D),
\]

and similarly for \(M_B\).

In [ ]:
p_om = P.sum(axis=1)
p_mb = P.sum(axis=0)

p_om /= p_om.sum()
p_mb /= p_mb.sum()

def discrete_quantile(grid, p, q):
    c = np.cumsum(p)
    return np.interp(q, c, grid)

om_q = [discrete_quantile(OMEGA_GRID,p_om,q) for q in (0.16,0.50,0.84)]
mb_q = [discrete_quantile(MB_GRID,p_mb,q) for q in (0.16,0.50,0.84)]

print("Omega_m 16/50/84:", om_q)
print("M_B     16/50/84:", mb_q)

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(OMEGA_GRID, p_om)
ax.set_xlabel(r"$\Omega_m$")
ax.set_ylabel("Marginal posterior probability")
ax.set_title(r"$p(\Omega_m\mid D)$")
plt.show()

### Exercise 2

Report the posterior median and central 68% credible interval for both parameters.

Then compare the posterior mode in \(\Omega_m\) with the profile-likelihood MLE from Lab 03.

Explain why they need not be numerically identical.

# Part III — Prior sensitivity

## 5. A deliberately informative alternative prior

To learn what prior sensitivity looks like, apply

\[
\Omega_m\sim\mathcal N(0.30,0.03^2)
\]

truncated to the grid.

This is a **teaching prior**, not a claim that this is the correct external prior to use in a publication.

In [ ]:
prior_om_alt = np.exp(-0.5*((OMEGA_GRID-0.30)/0.03)**2)
prior_om_alt /= prior_om_alt.sum()

log_prior2d = np.log(prior_om_alt[:,None] + 1e-300)
logP_alt = logL + log_prior2d
logP_alt -= logsumexp(logP_alt)
P_alt = np.exp(logP_alt)
P_alt /= P_alt.sum()

p_om_alt = P_alt.sum(axis=1)
p_om_alt /= p_om_alt.sum()

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(OMEGA_GRID, p_om, label="Baseline uniform prior")
ax.plot(OMEGA_GRID, p_om_alt, label="Informative teaching prior")
ax.set_xlabel(r"$\Omega_m$")
ax.set_ylabel("Marginal posterior probability")
ax.legend()
plt.show()

### Exercise 3

Calculate the 16th/50th/84th percentiles under the alternative prior.

Then answer:

1. Did the posterior location change?
2. Did its width change?
3. Is the data likelihood strong enough to overwhelm this prior?
4. Why is “the posterior is objective” an incorrect statement?

# Part IV — Posterior predictive thinking

## 6. Draw parameter pairs from the grid posterior

A posterior predictive check asks whether parameter values supported by the posterior generate predictions that resemble the observed data in relevant ways.

For this lab, draw posterior parameter pairs and calculate residual curves. We are **not** generating synthetic supernovae.

In [ ]:
n_draw = 100
flat_idx = rng.choice(P.size, size=n_draw, replace=True, p=P.ravel())
io, im = np.unravel_index(flat_idx, P.shape)

z_plot = np.geomspace(max(0.01,data["zHD"].min()), data["zHD"].max(), 250)

fig, ax = plt.subplots(figsize=(8,5))
for k in range(n_draw):
    om = OMEGA_GRID[io[k]]
    mb = MB_GRID[im[k]]
    m_pred = mu_flat_lcdm(z_plot, om, H0=H0_CONVENTION) + mb
    ax.plot(z_plot, m_pred, alpha=0.05)

ax.scatter(data["zHD"], data["m_b_corr"], s=5, alpha=0.25)
ax.set_xscale("log")
ax.set_xlabel(r"$z_{\rm HD}$")
ax.set_ylabel(r"$m_B^{\rm corr}$")
ax.set_title("Posterior-supported Hubble relations over real data")
plt.show()

### Exercise 4 — residual diagnostic

Using the posterior median parameters:

1. calculate magnitude residuals;
2. bin them only for visualization in 6–10 redshift bins;
3. plot median residual per bin with a bootstrap uncertainty on that median.

State clearly that these bins are a visualization diagnostic, not independent measurements used in the likelihood.

# 7. In-class checkpoint

Show:

1. normalized 2-D posterior;
2. 68% and 95% contours;
3. marginal \(p(\Omega_m\mid D)\);
4. 68% credible intervals;
5. prior-sensitivity comparison;
6. one posterior-supported Hubble-diagram visualization.

### Exit question

> Which statements in this lab are properties of the data, and which are conditional on the prior and model?

# 8. Bridge to Lab 05

In two dimensions, a grid is powerful because we can directly inspect and normalize the posterior.

Next week we deliberately introduce MCMC and ask:

- Can a sampler reproduce the known grid posterior?
- How do we diagnose convergence?
- What changes when systematic covariance is added?
- Why should we not use MCMC merely because it is fashionable?